# Observability: Evaluation Pipeline

----

This notebook demonstrates a **stage-by-stage evaluation pipeline** for a Semantic Kernel-based chat application, using Application Insights logs and the **Azure AI Foundry Evals API** (`azure-ai-projects>=2.0.1`).

You will learn how to:

- **Configure Application Insights** telemetry for Semantic Kernel
- **Build an SK orchestration layer**: intent classification → agent routing → function calling → answer generation
- **Parse Application Insights logs** into structured evaluation datasets (JSONL)
- **Register custom evaluators** in the Foundry evaluator catalog (code-based exact-match)
- **Run evaluations via Evals API**: custom evaluators (intent, agent, method) + builtin evaluators (groundedness, coherence, relevance)
- **Generate HTML dashboard** for comprehensive result visualization

## Table of Contents

- [Setup](#setup)
- [Part 1: Connect to SK Orchestrator Backend](#part-1-connect-to-sk-orchestrator-backend)
- [Part 2: Orchestration via SK Backend API](#part-2-orchestration-via-sk-backend-api)
- [Part 3: Build Evaluation Dataset from Golden Queries](#part-3-build-evaluation-dataset-from-golden-queries)
- [Part 4: Register Custom Evaluators in Foundry Catalog](#part-4-register-custom-evaluators-in-foundry-catalog)
- [Part 5: Run Evaluation via Evals API](#part-5-run-evaluation-via-evals-api)
- [Part 6: Evaluation Results](#part-6-evaluation-results)
- [Part 7: Evaluation Dashboard (HTML)](#part-7-evaluation-dashboard-html)
- [Wrap-up](#wrap-up)

## Setup

This notebook reuses the configuration file (`.foundry_config.json`) created by `0_setup/1_setup.ipynb`.

### Prerequisites

| Requirement | Description |
|-------------|-------------|
| `APPLICATIONINSIGHTS_CONNECTION_STRING` | Azure Application Insights connection string |
| `AZURE_OPENAI_ENDPOINT` | Azure OpenAI endpoint URL |
| `AZURE_OPENAI_API_KEY` | Azure OpenAI API key |
| `AZURE_OPENAI_CHAT_DEPLOYMENT_NAME` | Deployment name (e.g., `gpt-4.1`) |
| `AZURE_AI_PROJECT_ENDPOINT` | Azure AI Foundry project endpoint |
| `APPLICATIONINSIGHTS_RESOURCE_ID` | Application Insights resource ID (for trace-based evaluation) |
| `azure-ai-projects>=2.0.1` | Required SDK version for Foundry Evals API |
| **Azure AI Developer** role | Required on the Foundry resource for evaluator registration and eval creation |
| **Cognitive Services Contributor** role | Required on the Foundry resource for server-side eval runs (`temporaryDataReference`) |

> **Important**: To run evaluations via the Foundry Evals API (with portal `report_url`), your identity needs the following roles on the **Foundry resource** (`Microsoft.CognitiveServices/accounts`). Without these roles, the notebook falls back to local evaluation mode automatically.
>
> | Role | Purpose |
> |------|---------|
> | **Azure AI Developer** | Register custom evaluators, create eval objects, call OpenAI Evals API |
> | **Cognitive Services Contributor** | Write to internal asset store for eval run data (`temporaryDataReference`) |
>
> Assign via: **Azure Portal** → Foundry resource → **Access control (IAM)** → **Add role assignment** → select each role → select your user.
>
> In the new Foundry architecture, there is no separate storage account — the asset store is managed internally by the Foundry resource.

In [3]:
# Environment setup and PATH configuration
import json
import os
import subprocess
import csv
from typing import Dict, List, Any
from dataclasses import dataclass
from dotenv import load_dotenv

load_dotenv(override=True)

# Ensure the notebook kernel can find Azure CLI on PATH
possible_paths = [
    '/opt/homebrew/bin',
    '/usr/local/bin',
    '/usr/bin',
    '/home/linuxbrew/.linuxbrew/bin',
]

az_path = None
try:
    result = subprocess.run(['which', 'az'], capture_output=True, text=True)
    if result.returncode == 0:
        az_path = os.path.dirname(result.stdout.strip())
        print(f'🔍 Azure CLI found: {result.stdout.strip()}')
except Exception:
    pass

paths_to_add: list[str] = []
if az_path and az_path not in os.environ.get('PATH', ''):
    paths_to_add.append(az_path)
else:
    for path in possible_paths:
        if os.path.exists(path) and path not in os.environ.get('PATH', ''):
            paths_to_add.append(path)

if paths_to_add:
    os.environ['PATH'] = ':'.join(paths_to_add) + ':' + os.environ.get('PATH', '')
    print(f"✅ Added to PATH: {', '.join(paths_to_add)}")
else:
    print('✅ PATH looks good already')

🔍 Azure CLI found: /anaconda/envs/azureml_py38/bin//az
✅ Added to PATH: /home/linuxbrew/.linuxbrew/bin


In [4]:
# Load Foundry project settings
from azure.identity import DefaultAzureCredential

config_file = '../0_setup/.foundry_config.json'
try:
    with open(config_file, 'r', encoding='utf-8') as f:
        config = json.load(f)
except FileNotFoundError as e:
    print(f"⚠️ Could not find '{config_file}'.")
    print('💡 Run 0_setup/1_setup.ipynb first to create it.')
    raise e

FOUNDRY_NAME = config.get('FOUNDRY_NAME')
RESOURCE_GROUP = config.get('RESOURCE_GROUP')
LOCATION = config.get('LOCATION')
PROJECT_NAME = config.get('PROJECT_NAME', 'proj-default')
AZURE_AI_PROJECT_ENDPOINT = config.get('AZURE_AI_PROJECT_ENDPOINT')

AZURE_OPENAI_ENDPOINT = os.environ.get("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.environ.get("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_CHAT_DEPLOYMENT_NAME = os.environ.get(
    "AZURE_OPENAI_CHAT_DEPLOYMENT_NAME", "gpt-4.1"
)
AZURE_OPENAI_API_VERSION = os.environ.get(
    "AZURE_OPENAI_API_VERSION", "2025-03-01-preview"
)
CONNECTION_STRING = os.environ.get("APPLICATIONINSIGHTS_CONNECTION_STRING")

os.environ['FOUNDRY_NAME'] = FOUNDRY_NAME or ''
os.environ['LOCATION'] = LOCATION or ''
os.environ['RESOURCE_GROUP'] = RESOURCE_GROUP or ''
os.environ['AZURE_SUBSCRIPTION_ID'] = config.get('AZURE_SUBSCRIPTION_ID', '')

credential = DefaultAzureCredential()

print(f"✅ Loaded settings from '{config_file}'.")
print(f"📌 Foundry: {FOUNDRY_NAME} | Region: {LOCATION}")
print(f"📌 Project endpoint: {AZURE_AI_PROJECT_ENDPOINT}")
print(f"📌 Chat deployment: {AZURE_OPENAI_CHAT_DEPLOYMENT_NAME}")

✅ Loaded settings from '../0_setup/.foundry_config.json'.
📌 Foundry: foundry-rq90gs | Region: swedencentral
📌 Project endpoint: https://foundry-rq90gs.services.ai.azure.com/api/projects/default-project
📌 Chat deployment: gpt-4.1


/afh/code/agent-operator-lab/4_observability/sk_backend/.venv/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


## Part 1: Connect to SK Orchestrator Backend

The evaluation pipeline consumes the **SK Orchestrator Backend** (`sk_backend/`) — a FastAPI service that wraps Semantic Kernel, XML context loading, and OpenTelemetry tracing behind three endpoints:

| Endpoint | Method | Description |
|----------|--------|-------------|
| `/health` | `GET` | Service health check |
| `/classify` | `POST` | Intent classification only |
| `/chat` | `POST` | Full pipeline: intent → XML context → LLM answer |

```
┌─────────────────────────────────────────────────┐
│  This Notebook (Evaluation Pipeline)             │
│                                                  │
│  httpx ──► POST /classify  (intent only)         │
│        ──► POST /chat      (full pipeline)       │
│        ──► GET  /health    (connection check)    │
│                                                  │
│            ▼                                     │
│  ┌──────────────────────────────────────┐        │
│  │  SK Orchestrator Backend (FastAPI)    │        │
│  │  Semantic Kernel + OpenTelemetry      │        │
│  │  → Application Insights              │        │
│  └──────────────────────────────────────┘        │
└─────────────────────────────────────────────────┘
```

> **Prerequisite**: Kill any existing instance and (re)start the backend:
> ```bash
> kill $(lsof -ti :8000) 2>/dev/null; sleep 1
> cd 4_observability/sk_backend && nohup uvicorn sk_orchestrator.main:app --host 0.0.0.0 --port 8000 > sk_backend.log 2>&1 &
> ```


In [4]:
# Connect to SK Orchestrator Backend
import httpx

SK_BACKEND_URL = os.environ.get("SK_BACKEND_URL", "http://localhost:8000")

try:
    health = httpx.get(
        f"{SK_BACKEND_URL}/health", timeout=10.0
    ).json()
    print(f"✅ SK Backend connected: {SK_BACKEND_URL}")
    print(f"   Status:  {health['status']}")
    print(f"   Service: {health['service']}")
    print(f"   Model:   {health['model']}")
except httpx.ConnectError:
    raise ConnectionError(
        f"❌ Cannot connect to SK Backend at {SK_BACKEND_URL}\n"
        "   Start the backend first:\n"
        "   cd 4_observability/sk_backend && "
        "uvicorn sk_orchestrator.main:app --port 8000"
    )


✅ SK Backend connected: http://localhost:8000
   Status:  healthy
   Service: sk-orchestrator-backend
   Model:   gpt-4.1


## Part 2: Orchestration via SK Backend API

Instead of importing Semantic Kernel directly, this notebook calls the **SK Orchestrator Backend** API endpoints to classify intents and generate LLM answers grounded in XML context files.

### API Endpoints Used

| Endpoint | Purpose | Response Keys |
|----------|---------|---------------|
| `POST /classify` | Intent classification | `intent`, `confidence`, `reasoning` |
| `POST /chat` | Full pipeline (intent → XML context → LLM answer) | `query`, `intent`, `confidence`, `agent`, `method`, `context_source`, `answer` |

### Pipeline Flow (via API)

| Step | Description | API Call |
|------|-------------|---------|
| 1 | Intent Classification | `POST /classify` |
| 2 | XML Context Selection + LLM Answer | `POST /chat` |
| 3 | Evaluation Dataset Construction | Combine API responses with CSV ground truth |

### Agent → XML Context Mapping (handled by backend)

| Agent | Method | Intent | XML Context |
|-------|--------|--------|-------------|
| `productAgent` | `search_products` | `product_search` | `product_contexts.xml` |
| `recommendAgent` | `search_recsys` | `recommendation` | `recommendation_contexts.xml` |
| `policyAgent` | `search_policy` | `policy` | (default response) |


In [5]:
# Intent → Agent mapping (ground truth from logs, used by evaluators)
INTENT_AGENT_MAP: Dict[str, Dict[str, str]] = {
    "product_search": {
        "agent": "productAgent",
        "method": "search_products",
    },
    "recommendation": {
        "agent": "recommendAgent",
        "method": "search_recsys",
    },
    "policy": {
        "agent": "policyAgent",
        "method": "search_policy",
    },
    "beauty": {
        "agent": "beautyAgent",
        "method": "search_beauty",
    },
}

print("✅ Intent → Agent mapping defined")
print(f"   Supported intents: {list(INTENT_AGENT_MAP.keys())}")


✅ Intent → Agent mapping defined
   Supported intents: ['product_search', 'recommendation', 'policy', 'beauty']


In [6]:
# API wrapper functions for SK Backend


async def api_classify(query: str) -> dict:
    """
    Call POST /classify to classify user intent.

    Parameters:
    query (str): User query text.

    Returns:
    dict: {"intent", "confidence", "reasoning"}
    """
    async with httpx.AsyncClient(timeout=120.0) as client:
        resp = await client.post(
            f"{SK_BACKEND_URL}/classify",
            json={"query": query},
        )
        resp.raise_for_status()
        return resp.json()


async def api_chat(query: str) -> dict:
    """
    Call POST /chat to run the full pipeline.

    Parameters:
    query (str): User query text.

    Returns:
    dict: {"query", "intent", "confidence", "agent",
           "method", "context_source", "answer"}
    """
    async with httpx.AsyncClient(timeout=120.0) as client:
        resp = await client.post(
            f"{SK_BACKEND_URL}/chat",
            json={"query": query},
        )
        resp.raise_for_status()
        return resp.json()


print("✅ API wrapper functions defined")
print(f"   api_classify(query) → POST {SK_BACKEND_URL}/classify")
print(f"   api_chat(query)     → POST {SK_BACKEND_URL}/chat")


✅ API wrapper functions defined
   api_classify(query) → POST http://localhost:8000/classify
   api_chat(query)     → POST http://localhost:8000/chat


In [7]:
# Test POST /classify endpoint
test_classify = await api_classify(
    "이니스프리 수퍼 화산송이 모공 마스크는?"
)

print("🧪 /classify Endpoint Test")
print(f"   Intent:     {test_classify['intent']}")
print(f"   Confidence: {test_classify['confidence']:.2f}")
print(f"   Reasoning:  {test_classify['reasoning']}")


🧪 /classify Endpoint Test
   Intent:     product_search
   Confidence: 0.95
   Reasoning:  The user is asking about a specific product, '이니스프리 수퍼 화산송이 모공 마스크', which indicates a product search intent.


In [8]:
# Test POST /chat endpoint
test_chat = await api_chat(
    "여드름 피부에 좋은 제품 추천해줘"
)

print("🧪 /chat Endpoint Test")
print(f"   Intent:  {test_chat['intent']}")
print(f"   Agent:   {test_chat['agent']}.{test_chat['method']}")
print(f"   Context: {test_chat['context_source']}")
print(f"   Answer:  {test_chat['answer'][:200]}...")


🧪 /chat Endpoint Test
   Intent:  recommendation
   Agent:   recommendAgent.search_recsys
   Context: /afh/code/agent-operator-lab/4_observability/contexts/recommendation_contexts.xml
   Answer:  여드름 피부(트러블성 피부)에 가장 적합한 제품은 아래와 같습니다:

**1. 코스알엑스 AHA/BHA 클래리파잉 트리트먼트 토너**
- **추천 이유:**  
  이 제품은 AHA와 BHA 성분이 저농도로 함유되어 있어 각질과 모공 속 피지를 동시에 케어해줍니다. 트러블성 피부의 피부결 개선과 피지 조절에 탁월하며, 매일 사용해도 자극이 적어 민감한 여드...


In [9]:
# Orchestration wrappers — match the original pipeline function signatures


async def classify_intent(query: str) -> dict:
    """
    Classify intent via SK Backend API.

    Parameters:
    query (str): User query text.

    Returns:
    dict: Intent classification result from POST /classify.
    """
    return await api_classify(query)


async def run_pipeline(query: str) -> dict:
    """
    Run the full pipeline via SK Backend API.

    Parameters:
    query (str): User query text.

    Returns:
    dict: Full pipeline result from POST /chat.
    """
    return await api_chat(query)


print("✅ Orchestration wrappers defined")
print("   classify_intent(query) → POST /classify")
print("   run_pipeline(query)    → POST /chat")


✅ Orchestration wrappers defined
   classify_intent(query) → POST /classify
   run_pipeline(query)    → POST /chat


In [10]:
# Load golden query set
GOLDEN_QUERIES_PATH = "./log/golden_user_query_list.jsonl"

golden_queries: List[Dict[str, str]] = []
with open(GOLDEN_QUERIES_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        golden_queries.append(json.loads(line.strip()))

print(f"✅ Loaded {len(golden_queries)} golden queries")
print(f"   from {GOLDEN_QUERIES_PATH}")

from collections import Counter
intent_dist = Counter(q["expected_intent"] for q in golden_queries)
for intent, count in intent_dist.most_common():
    print(f"   {intent}: {count}")

print(f"\n--- Sample Queries ---")
for q in golden_queries[:3]:
    print(f"   [{q['expected_intent']:18s}] {q['query'][:60]}")


✅ Loaded 50 golden queries
   from ./log/golden_user_query_list.jsonl
   product_search: 20
   recommendation: 15
   policy: 10
   beauty: 5

--- Sample Queries ---
   [product_search    ] 마몽드 플래시토닝 데이지 리퀴드 마스크는 어떤 피부 타입에 좋아?
   [product_search    ] 이니스프리 수퍼 화산송이 모공 마스크 성분이 궁금해요
   [product_search    ] 설화수 윤조에센스 사용방법 알려줘


## Part 3: Build Evaluation Dataset from Golden Queries

Sends the golden query set (`golden_user_query_list.jsonl`) to the SK Backend via `POST /chat`, collects the LLM responses along with the XML context used, and saves the complete evaluation dataset as `llm_result_list.jsonl`.

### Pipeline

```
golden_user_query_list.jsonl     POST /chat (SK Backend)     llm_result_list.jsonl
─────────────────────────── ──► ───────────────────────── ──► ──────────────────────
50 queries with labels           Intent → XML → LLM          Predictions + context
(expected_intent/agent/method)                                + response for eval
```

### Output Fields in `llm_result_list.jsonl`

| Field | Source | Used By |
|-------|--------|---------|
| `query` | Golden query set | All evaluators |
| `expected_intent/agent/method` | Golden labels | Step 1-3 custom evaluators |
| `predicted_intent/agent/method` | SK Backend `/chat` response | Step 1-3 custom evaluators |
| `context` | XML file matching predicted intent | Retrieval + Groundedness |
| `response` | LLM-generated answer | Groundedness + Similarity |
| `ground_truth` | (empty — no human-written answer) | Similarity (skipped if empty) |


In [11]:
# Load XML context files for evaluation (same files the backend uses)
import xml.etree.ElementTree as ET

PRODUCT_CONTEXTS_PATH = "./contexts/product_contexts.xml"
RECOMMENDATION_CONTEXTS_PATH = "./contexts/recommendation_contexts.xml"

_xml_contexts: Dict[str, str] = {}
for label, path in [
    ("product_search", PRODUCT_CONTEXTS_PATH),
    ("recommendation", RECOMMENDATION_CONTEXTS_PATH),
]:
    tree = ET.parse(path)
    _xml_contexts[label] = ET.tostring(
        tree.getroot(), encoding="unicode"
    )

print("✅ XML context files loaded for evaluation")
for k, v in _xml_contexts.items():
    print(f"   {k}: {len(v):,} chars")


✅ XML context files loaded for evaluation
   product_search: 20,275 chars
   recommendation: 28,297 chars


In [12]:
# Run golden queries through SK Backend and save results
LLM_RESULT_PATH = "./log/llm_result_list.jsonl"

print(f"🔄 Sending {len(golden_queries)} golden queries to SK Backend...")

llm_results: List[Dict[str, Any]] = []
# adjust the number of list
count = 5
run_count = min(count, len(golden_queries))  # 예: count=5면 5개만 실행
target_queries = golden_queries[:run_count]

print(f"🔄 Running {run_count}/{len(golden_queries)} queries...")

for i, gq in enumerate(target_queries):
    try:
        chat_result = await api_chat(gq["query"])
        predicted_intent = chat_result["intent"]
        predicted_agent = chat_result["agent"]
        predicted_method = chat_result["method"]
        llm_response = chat_result["answer"]

        # XML context matching the predicted intent
        xml_context = _xml_contexts.get(predicted_intent, "")
    except Exception as e:
        print(f"   ⚠️ [{i+1}] Error: {e}")
        predicted_intent = "unknown"
        predicted_agent = "unknown"
        predicted_method = "unknown"
        llm_response = ""
        xml_context = ""

    llm_results.append({
        "query": gq["query"],
        # Golden labels (ground truth)
        "expected_intent": gq["expected_intent"],
        "expected_agent": gq["expected_agent"],
        "expected_method": gq["expected_method"],
        # SK Backend predictions
        "predicted_intent": predicted_intent,
        "predicted_agent": predicted_agent,
        "predicted_method": predicted_method,
        # XML context for Retrieval/Groundedness
        "context": xml_context,
        # LLM answer
        "response": llm_response,
        # No human-written ground truth
        "ground_truth": "",
    })

    if (i + 1) % 10 == 0 or (i + 1) == run_count:
        print(f"   Processed {i + 1}/{run_count}")

# Save results
with open(LLM_RESULT_PATH, 'w', encoding='utf-8') as f:
    for item in llm_results:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

filled = sum(1 for r in llm_results if r["context"])
print(f"\n✅ {len(llm_results)} results saved to {LLM_RESULT_PATH}")
print(f"   Records with XML context: {filled}/{len(llm_results)}")


🔄 Sending 50 golden queries to SK Backend...
🔄 Running 5/50 queries...


   Processed 5/5

✅ 5 results saved to ./log/llm_result_list.jsonl
   Records with XML context: 3/5


In [13]:
# Verify the evaluation dataset
EVAL_DATA_PATH = "./log/llm_result_list.jsonl"

print(f"📊 Evaluation dataset: {EVAL_DATA_PATH}")
print(f"   Total records: {len(llm_results)}")

from collections import Counter

print("\nPredicted intent distribution:")
pred_dist = Counter(r["predicted_intent"] for r in llm_results)
for intent, count in pred_dist.most_common():
    print(f"   {intent}: {count}")

print("\nIntent match preview (first 10):")
for r in llm_results[:10]:
    match = "✅" if (
        r["predicted_intent"] == r["expected_intent"]
    ) else "❌"
    print(
        f"   {match} expected={r['expected_intent']:18s}"
        f" predicted={r['predicted_intent']:18s}"
        f" | {r['query'][:50]}"
    )


📊 Evaluation dataset: ./log/llm_result_list.jsonl
   Total records: 5

Predicted intent distribution:
   product_search: 3
   beauty: 2

Intent match preview (first 10):
   ✅ expected=product_search     predicted=product_search     | 마몽드 플래시토닝 데이지 리퀴드 마스크는 어떤 피부 타입에 좋아?
   ✅ expected=product_search     predicted=product_search     | 이니스프리 수퍼 화산송이 모공 마스크 성분이 궁금해요
   ❌ expected=product_search     predicted=beauty             | 설화수 윤조에센스 사용방법 알려줘
   ✅ expected=product_search     predicted=product_search     | 라네즈 워터 슬리핑 마스크 가격이 얼마야?
   ❌ expected=product_search     predicted=beauty             | 헤라 블랙 쿠션 21N1 바닐라 호수가 나한테 맞을까?


## Part 4: Register Custom Evaluators in Foundry Catalog

Register **code-based custom evaluators** in the Azure AI Foundry evaluator catalog using `project_client.beta.evaluators.create_version()`. These evaluators are executed server-side by the Evals API.

| Evaluator | Type | Logic | Metric |
|-----------|------|-------|--------|
| `intent_accuracy` | Code-based | `predicted_intent == expected_intent` | 0.0 or 1.0 |
| `agent_relevance` | Code-based | `predicted_agent == expected_agent` | 0.0 or 1.0 |
| `method_relevance` | Code-based | `predicted_method == expected_method` | 0.0 or 1.0 |

### SDK APIs Used

```
project_client.beta.evaluators.create_version()   → Register evaluator in catalog
openai_client.evals.create()                       → Create evaluation group
openai_client.evals.runs.create()                  → Run evaluation with JSONL data
openai_client.evals.runs.output_items.list()       → Retrieve per-row results
```

In [20]:
# Register code-based custom evaluators in Foundry Catalog
from datetime import datetime
import time
from pprint import pprint

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    CodeBasedEvaluatorDefinition,
    EvaluatorVersion,
    EvaluatorMetric,
    EvaluatorMetricType,
    EvaluatorMetricDirection,
)
from openai.types.eval_create_params import DataSourceConfigCustom
from openai.types.evals.create_eval_jsonl_run_data_source_param import (
    CreateEvalJSONLRunDataSourceParam,
    SourceFileContent,
    SourceFileContentContent,
)

EVAL_UUID = datetime.now().strftime("%Y%m%d%H%M%S")
MODEL_DEPLOYMENT = AZURE_OPENAI_CHAT_DEPLOYMENT_NAME
EVAL_DATA_PATH = "./log/llm_result_list.jsonl"
MAX_CONTEXT_CHARS = 8000

RESULT_METRIC = {
    "result": EvaluatorMetric(
        type=EvaluatorMetricType.ORDINAL,
        desirable_direction=EvaluatorMetricDirection.INCREASE,
        min_value=0.0,
        max_value=1.0,
    )
}


def _exact_match_code(pred: str, exp: str) -> str:
    """Generate Python code for an exact-match evaluator.

    grade() must take exactly 2 args: item, sample.
    """
    return (
        "def grade(item, sample):\n"
        f"    p = (item.get('{pred}', '') or '')"
        ".strip().lower()\n"
        f"    e = (item.get('{exp}', '') or '')"
        ".strip().lower()\n"
        "    return 1.0 if p == e else 0.0\n"
    )


def _code_evaluator_version(
    code: str,
    fields: list[str],
    display_name: str,
    description: str,
) -> EvaluatorVersion:
    """Build EvaluatorVersion for code-based evaluator."""
    return EvaluatorVersion(
        display_name=display_name,
        description=description,
        definition=CodeBasedEvaluatorDefinition(
            code_text=code,
            data_schema={
                "required": ["item"],
                "type": "object",
                "properties": {
                    "item": {
                        "type": "object",
                        "properties": {
                            f: {"type": "string"}
                            for f in fields
                        },
                    },
                },
            },
            metrics=RESULT_METRIC,
        ),
    )


# Initialize Foundry clients (kept open for Parts 4-7)
eval_credential = DefaultAzureCredential()
project_client = AIProjectClient(
    endpoint=AZURE_AI_PROJECT_ENDPOINT,
    credential=eval_credential,
)
openai_client = project_client.get_openai_client()

# Evaluator specs: (name, predicted_field, expected_field, display_name, description)
EVALUATOR_SPECS = [
    (
        "intent_accuracy",
        "predicted_intent",
        "expected_intent",
        "Intent Accuracy",
        "Exact-match evaluator that checks whether the predicted intent matches the expected intent. Returns 1.0 if they match, 0.0 otherwise.",
    ),
    (
        "agent_relevance",
        "predicted_agent",
        "expected_agent",
        "Agent Relevance",
        "Exact-match evaluator that checks whether the predicted agent matches the expected agent. Returns 1.0 if they match, 0.0 otherwise.",
    ),
    (
        "method_relevance",
        "predicted_method",
        "expected_method",
        "Method Relevance",
        "Exact-match evaluator that checks whether the predicted method matches the expected method. Returns 1.0 if they match, 0.0 otherwise.",
    ),
]

# Register each evaluator in Foundry Catalog
registered_evaluators = {}
for name, pred_f, exp_f, disp_name, desc in EVALUATOR_SPECS:
    full_name = f"{name}_{EVAL_UUID}"
    code = _exact_match_code(pred_f, exp_f)
    ev = _code_evaluator_version(
        code, [pred_f, exp_f], disp_name, desc,
    )
    reg = project_client.beta.evaluators.create_version(
        name=full_name, evaluator_version=ev,
    )
    registered_evaluators[name] = reg
    print(
        f"✅ {name}: {reg.name} (version {reg.version})"
    )
    print(f"   Display name: {disp_name}")
    print(f"   Description:  {desc[:60]}...")

print(
    f"\n📋 {len(registered_evaluators)} custom evaluators "
    f"registered in Foundry Catalog"
)

✅ intent_accuracy: intent_accuracy_20260320045247 (version 1)
   Display name: Intent Accuracy
   Description:  Exact-match evaluator that checks whether the predicted inte...
✅ agent_relevance: agent_relevance_20260320045247 (version 1)
   Display name: Agent Relevance
   Description:  Exact-match evaluator that checks whether the predicted agen...
✅ method_relevance: method_relevance_20260320045247 (version 1)
   Display name: Method Relevance
   Description:  Exact-match evaluator that checks whether the predicted meth...

📋 3 custom evaluators registered in Foundry Catalog


## Part 5: Run Evaluation via Evals API

Create **one unified evaluation group** with 6 testing criteria, then run it against the `llm_result_list.jsonl` dataset using inline JSONL data.

### Testing Criteria (6 Evaluators)

| # | Evaluator | Type | Data Mapping | Purpose |
|---|-----------|------|-------------|---------|
| 1 | `intent_accuracy` | Custom (code) | predicted/expected intent | Intent classification accuracy |
| 2 | `agent_relevance` | Custom (code) | predicted/expected agent | Agent routing accuracy |
| 3 | `method_relevance` | Custom (code) | predicted/expected method | Method selection accuracy |
| 4 | `groundedness` | Builtin | query, response, context | Response grounded in retrieved context |
| 5 | `coherence` | Builtin | query, response | Response coherence quality |
| 6 | `relevance` | Builtin | query, response | Response relevance to query |

### Data Flow

```text
llm_result_list.jsonl
       │
       ▼
CreateEvalJSONLRunDataSourceParam (inline JSONL)
       │
       ▼
openai_client.evals.runs.create()
       │
       ├──► Custom evaluators (Steps 1-3): exact-match scoring
       └──► Builtin evaluators (Step 4): LLM-based quality scoring
       │
       ▼
Foundry Portal → report_url
```

In [21]:
# Create evaluation + try run via Evals API (with fallback)
from openai.types.evals.create_eval_jsonl_run_data_source_param import (
    SourceFileID,
)

# 1. Load evaluation dataset from JSONL
eval_records = []
with open(EVAL_DATA_PATH, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            eval_records.append(json.loads(line.strip()))

print(f"📊 Loaded {len(eval_records)} records from {EVAL_DATA_PATH}")

# 2. Define item schema
data_source_config = DataSourceConfigCustom(
    type="custom",
    item_schema={
        "type": "object",
        "properties": {
            "query": {"type": "string"},
            "expected_intent": {"type": "string"},
            "predicted_intent": {"type": "string"},
            "expected_agent": {"type": "string"},
            "predicted_agent": {"type": "string"},
            "expected_method": {"type": "string"},
            "predicted_method": {"type": "string"},
            "context": {"type": "string"},
            "response": {"type": "string"},
            "ground_truth": {"type": "string"},
        },
        "required": ["query"],
    },
)

# 3. Build testing criteria
testing_criteria = []

for eval_name, evaluator in registered_evaluators.items():
    spec = next(s for s in EVALUATOR_SPECS if s[0] == eval_name)
    _, pred_field, exp_field, _, _ = spec
    testing_criteria.append({
        "type": "azure_ai_evaluator",
        "name": eval_name,
        "evaluator_name": evaluator.name,
        "pass_threshold": 0.5,
        "initialization_parameters": {
            "pass_threshold": 0.5,
        },
        "data_mapping": {
            pred_field: f"{{{{item.{pred_field}}}}}",
            exp_field: f"{{{{item.{exp_field}}}}}",
        },
    })

BUILTIN_EVALUATORS = [
    {
        "name": "groundedness",
        "evaluator_name": "builtin.groundedness",
        "data_mapping": {
            "query": "{{item.query}}",
            "response": "{{item.response}}",
            "context": "{{item.context}}",
        },
    },
    {
        "name": "coherence",
        "evaluator_name": "builtin.coherence",
        "data_mapping": {
            "query": "{{item.query}}",
            "response": "{{item.response}}",
        },
    },
    {
        "name": "relevance",
        "evaluator_name": "builtin.relevance",
        "data_mapping": {
            "query": "{{item.query}}",
            "response": "{{item.response}}",
            "context": "{{item.context}}",
        },
    },
]

for builtin in BUILTIN_EVALUATORS:
    testing_criteria.append({
        "type": "azure_ai_evaluator",
        **builtin,
        "pass_threshold": 0.5,
        "initialization_parameters": {
            "deployment_name": MODEL_DEPLOYMENT,
        },
    })

print(f"📋 Testing Criteria ({len(testing_criteria)} evaluators):")
for tc in testing_criteria:
    print(f"   • {tc['name']} → {tc['evaluator_name']} (pass_threshold={tc['pass_threshold']})")

# 4. Create evaluation object
eval_object = openai_client.evals.create(
    name=f"SK Pipeline Eval {EVAL_UUID}",
    data_source_config=data_source_config,
    testing_criteria=testing_criteria,  # type: ignore
)
print(f"\n📝 Evaluation created: {eval_object.id}")

# 5. Prepare JSONL and try creating eval run
EVAL_UPLOAD_PATH = "./log/eval_upload.jsonl"
with open(EVAL_UPLOAD_PATH, "w", encoding="utf-8") as f:
    for rec in eval_records:
        ctx = rec.get("context", "")
        if len(ctx) > MAX_CONTEXT_CHARS:
            ctx = ctx[:MAX_CONTEXT_CHARS]
        row = {
            "item": {
                "query": rec["query"],
                "expected_intent": rec["expected_intent"],
                "predicted_intent": rec["predicted_intent"],
                "expected_agent": rec["expected_agent"],
                "predicted_agent": rec["predicted_agent"],
                "expected_method": rec["expected_method"],
                "predicted_method": rec["predicted_method"],
                "context": ctx,
                "response": rec["response"],
                "ground_truth": rec.get("ground_truth", ""),
            }
        }
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"   📄 JSONL: {EVAL_UPLOAD_PATH}")

eval_run = None
foundry_mode = False

try:
    with open(EVAL_UPLOAD_PATH, "rb") as f:
        uploaded_file = openai_client.files.create(
            file=f, purpose="evals",
        )
    print(f"   📤 File uploaded: {uploaded_file.id}")

    eval_run = openai_client.evals.runs.create(
        eval_id=eval_object.id,
        name=f"SK Pipeline Run {EVAL_UUID}",
        metadata={
            "pipeline": "sk_orchestrator",
            "model": MODEL_DEPLOYMENT,
        },
        data_source=CreateEvalJSONLRunDataSourceParam(
            type="jsonl",
            source=SourceFileID(
                type="file_id", id=uploaded_file.id,
            ),
        ),
    )
    foundry_mode = True
    print(f"\n🚀 Eval run created: {eval_run.id}")

except Exception as e:
    error_msg = str(e)
    if "403" in error_msg or "temporaryDataReferenc" in error_msg:
        print(
            "\n⚠️ Eval run creation blocked by "
            "assetstore permission (403)."
        )
        print(
            "   → Falling back to local evaluation."
        )
        print(
            "   → Fix: Assign 'Cognitive Services "
            "Contributor' + 'Azure AI Developer' "
            "on the Foundry resource."
        )
    else:
        print(f"\n⚠️ Eval run failed: {error_msg[:100]}")
    foundry_mode = False

print(f"\n📌 Mode: {'Foundry (server-side)' if foundry_mode else 'Local evaluation'}")

📊 Loaded 5 records from ./log/llm_result_list.jsonl
📋 Testing Criteria (6 evaluators):
   • intent_accuracy → intent_accuracy_20260320045247 (pass_threshold=0.5)
   • agent_relevance → agent_relevance_20260320045247 (pass_threshold=0.5)
   • method_relevance → method_relevance_20260320045247 (pass_threshold=0.5)
   • groundedness → builtin.groundedness (pass_threshold=0.5)
   • coherence → builtin.coherence (pass_threshold=0.5)
   • relevance → builtin.relevance (pass_threshold=0.5)



📝 Evaluation created: eval_08791a5f8b2b4ba3a685fceae425ca7b
   📄 JSONL: ./log/eval_upload.jsonl
   📤 File uploaded: file-7cc3331d865d45bfae5b121e380fc7ad

🚀 Eval run created: evalrun_8f0d38c2244b44f1ac450510d6c9c1d9

📌 Mode: Foundry (server-side)


### Evaluation Execution

**Foundry mode**: Polls the server-side eval run until completion, then retrieves per-row output items.

**Local mode**: Runs evaluators locally — exact-match for Steps 1-3, LLM-based scoring for Step 4 (groundedness, coherence, relevance) using the Azure OpenAI deployment directly.

In [22]:
# Execute evaluation (Foundry or Local)
from collections import defaultdict
import re

output_items = []
eval_summary: Dict[str, List[float]] = defaultdict(list)
eval_rows: List[Dict[str, Any]] = []

if foundry_mode and eval_run:
    # ── Foundry mode: poll server-side run ──
    print("⏳ Waiting for Foundry eval run to complete...")
    while eval_run.status not in (
        "completed", "failed", "canceled",
    ):
        time.sleep(5)
        eval_run = openai_client.evals.runs.retrieve(
            run_id=eval_run.id,
            eval_id=eval_object.id,
        )
        print(f"   Status: {eval_run.status}")

    if eval_run.status == "completed":
        print(f"\n✅ Foundry evaluation completed!")
        output_items = list(
            openai_client.evals.runs.output_items.list(
                run_id=eval_run.id,
                eval_id=eval_object.id,
            )
        )
        if eval_run.report_url:
            print(f"   📊 Report: {eval_run.report_url}")
    else:
        print(f"❌ Foundry eval {eval_run.status}")
        foundry_mode = False

if not foundry_mode:
    # ── Local mode ──
    print("🔄 Running local evaluation...")

    def _exact_match(predicted: str, expected: str) -> float:
        """Exact-match score for Steps 1-3."""
        p = (predicted or "").strip().lower()
        e = (expected or "").strip().lower()
        return 1.0 if p == e else 0.0

    aoai_client = openai_client  # Reuse Foundry OpenAI client

    QUALITY_PROMPT = (
        "You are an evaluation judge. "
        "Score the assistant's response on **{metric}** "
        "on a scale of 1 to 5.\n\n"
        "### Query\n{query}\n\n"
        "{context_block}"
        "### Response\n{response}\n\n"
        "Return ONLY valid JSON: "
        '```json\n{{"score": <1-5>, '
        '"reason": "<brief>"}}\n```'
    )

    def _llm_quality_score(
        query: str,
        response: str,
        context: str,
        metric: str,
    ) -> float:
        """Score quality using LLM-as-judge."""
        ctx_block = ""
        if context:
            ctx_block = (
                f"### Context\n{context[:3000]}\n\n"
            )
        prompt = QUALITY_PROMPT.format(
            metric=metric,
            query=query,
            context_block=ctx_block,
            response=response[:2000],
        )
        try:
            result = aoai_client.chat.completions.create(
                model=MODEL_DEPLOYMENT,
                messages=[
                    {"role": "user", "content": prompt},
                ],
                temperature=0.0,
                max_tokens=150,
            )
            text = result.choices[0].message.content or ""
            # Extract score from JSON
            match = re.search(
                r'"score"\s*:\s*(\d+)', text,
            )
            if match:
                raw = int(match.group(1))
                return min(raw, 5) / 5.0
            # Fallback: look for bare number
            match = re.search(r'\b([1-5])\b', text)
            if match:
                return int(match.group(1)) / 5.0
        except Exception as exc:
            print(f"      ⚠️ {metric}: {str(exc)[:80]}")
        return 0.0

    for i, rec in enumerate(eval_records):
        row: Dict[str, Any] = {"query": rec["query"]}

        # Steps 1-3
        scores = {
            "intent_accuracy": _exact_match(
                rec["predicted_intent"],
                rec["expected_intent"],
            ),
            "agent_relevance": _exact_match(
                rec["predicted_agent"],
                rec["expected_agent"],
            ),
            "method_relevance": _exact_match(
                rec["predicted_method"],
                rec["expected_method"],
            ),
        }

        # Step 4: LLM quality
        ctx = rec.get("context", "")
        resp = rec.get("response", "")
        if resp:
            for metric in ("groundedness", "coherence", "relevance"):
                scores[metric] = _llm_quality_score(
                    rec["query"], resp, ctx, metric,
                )
        else:
            for metric in ("groundedness", "coherence", "relevance"):
                scores[metric] = 0.0

        row.update(scores)
        row["expected_intent"] = rec["expected_intent"]
        row["predicted_intent"] = rec["predicted_intent"]
        eval_rows.append(row)

        for k, v in scores.items():
            eval_summary[k].append(v)

        match_info = " ".join(
            f"{'✅' if v >= 0.5 else '❌'}{k[:3]}"
            for k, v in scores.items()
        )
        print(f"   [{i+1}/{len(eval_records)}] {match_info}")

    print(f"\n✅ Local evaluation: {len(eval_rows)} rows")

# Parse Foundry output_items if in Foundry mode
if foundry_mode and output_items:
    for item in output_items:
        row_data: Dict[str, Any] = {}
        if hasattr(item, "datasource_item"):
            ds = item.datasource_item or {}
            row_data["query"] = ds.get("query", "")
            row_data["expected_intent"] = ds.get(
                "expected_intent", ""
            )
            row_data["predicted_intent"] = ds.get(
                "predicted_intent", ""
            )
        if hasattr(item, "results") and item.results:
            results = item.results
            # results can be a list or a dict
            if isinstance(results, list):
                for r in results:
                    name = getattr(r, "name", None)
                    score = getattr(r, "score", None)
                    if name is None and isinstance(r, dict):
                        name = r.get("name")
                        score = r.get("score")
                    if name and score is not None:
                        try:
                            score = float(score)
                        except (ValueError, TypeError):
                            score = None
                        if score is not None:
                            eval_summary[name].append(score)
                            row_data[name] = score
            elif isinstance(results, dict):
                for name, result in results.items():
                    score = None
                    if hasattr(result, "score"):
                        score = result.score
                    elif isinstance(result, dict):
                        score = result.get("score")
                    if score is not None:
                        try:
                            score = float(score)
                        except (ValueError, TypeError):
                            score = None
                        if score is not None:
                            eval_summary[name].append(score)
                            row_data[name] = score
        eval_rows.append(row_data)

# Display summary
print("\n" + "=" * 60)
print("📊 Evaluation Summary")
print("=" * 60)
for name, scores in eval_summary.items():
    avg = sum(scores) / len(scores) if scores else 0.0
    print(f"   {name:25s} │ avg={avg:.3f}  n={len(scores)}")
print(f"   Total rows: {len(eval_rows)}")

⏳ Waiting for Foundry eval run to complete...


   Status: in_progress
   Status: in_progress
   Status: in_progress
   Status: in_progress
   Status: in_progress
   Status: in_progress
   Status: in_progress
   Status: in_progress
   Status: in_progress
   Status: in_progress
   Status: in_progress
   Status: completed

✅ Foundry evaluation completed!
   📊 Report: https://ai.azure.com/nextgen/r/PU090HnUQM-pTrQVSBLGyg,foundry-rg,,foundry-rq90gs,default-project/build/evaluations/eval_08791a5f8b2b4ba3a685fceae425ca7b/run/evalrun_8f0d38c2244b44f1ac450510d6c9c1d9

📊 Evaluation Summary
   intent_accuracy           │ avg=1.000  n=5
   agent_relevance           │ avg=1.000  n=5
   method_relevance          │ avg=1.000  n=5
   Total rows: 5


## Part 6: Evaluation Results

Parse the output items from the eval run into a structured summary. Each output item contains per-evaluator scores for one data row.

### Foundry Portal

The `report_url` links directly to the evaluation run in the Azure AI Foundry portal, where you can:

- View per-evaluator metric distributions
- Inspect individual row scores
- Compare across multiple evaluation runs

In [17]:
# Save evaluation summary to JSON
eval_summary_payload = {
    "eval_id": eval_object.id,
    "run_id": eval_run.id if eval_run else None,
    "status": (
        eval_run.status if eval_run else "local_completed"
    ),
    "report_url": (
        eval_run.report_url
        if eval_run and hasattr(eval_run, "report_url")
        else None
    ),
    "mode": "foundry" if foundry_mode else "local",
    "model": MODEL_DEPLOYMENT,
    "metrics": {
        k: {
            "avg": sum(v) / len(v) if v else 0.0,
            "count": len(v),
        }
        for k, v in eval_summary.items()
    },
    "timestamp": EVAL_UUID,
    "total_rows": len(eval_rows),
}

summary_path = "./log/eval_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(
        eval_summary_payload, f, ensure_ascii=False, indent=2,
    )

print(f"💾 Summary saved to {summary_path}")
display(eval_summary_payload)

💾 Summary saved to ./log/eval_summary.json


{'eval_id': 'eval_da2da4284a304a7eb6fa0916fddb687e',
 'run_id': 'evalrun_647f78afa73548f490ff0e357e2e57ba',
 'status': 'failed',
 'report_url': 'https://ai.azure.com/nextgen/r/PU090HnUQM-pTrQVSBLGyg,foundry-rg,,foundry-rq90gs,default-project/build/evaluations/eval_da2da4284a304a7eb6fa0916fddb687e/run/evalrun_647f78afa73548f490ff0e357e2e57ba',
 'mode': 'local',
 'model': 'gpt-4.1',
 'metrics': {'intent_accuracy': {'avg': 0.6, 'count': 5},
  'agent_relevance': {'avg': 0.6, 'count': 5},
  'method_relevance': {'avg': 0.6, 'count': 5},
  'groundedness': {'avg': 0.8400000000000001, 'count': 5},
  'coherence': {'avg': 0.8400000000000001, 'count': 5},
  'relevance': {'avg': 0.72, 'count': 5}},
 'timestamp': '20260320012624',
 'total_rows': 5}

In [18]:
# Per-row result inspection
print("=" * 60)
print("🔍 Per-Row Evaluation Details (first 5)")
print("=" * 60)

for i, row in enumerate(eval_rows[:5], 1):
    query = row.get("query", "")[:60]
    print(f"\n[{i}] {query}")
    for key, val in row.items():
        if key == "query":
            continue
        if isinstance(val, float):
            marker = "✅" if val >= 0.5 else "❌"
            print(f"    {marker} {key}: {val:.3f}")
        else:
            print(f"       {key}: {val}")

🔍 Per-Row Evaluation Details (first 5)

[1] 마몽드 플래시토닝 데이지 리퀴드 마스크는 어떤 피부 타입에 좋아?
    ✅ intent_accuracy: 1.000
    ✅ agent_relevance: 1.000
    ✅ method_relevance: 1.000
    ✅ groundedness: 1.000
    ✅ coherence: 1.000
    ✅ relevance: 1.000
       expected_intent: product_search
       predicted_intent: product_search

[2] 이니스프리 수퍼 화산송이 모공 마스크 성분이 궁금해요
    ✅ intent_accuracy: 1.000
    ✅ agent_relevance: 1.000
    ✅ method_relevance: 1.000
    ✅ groundedness: 1.000
    ✅ coherence: 0.800
    ❌ relevance: 0.400
       expected_intent: product_search
       predicted_intent: product_search

[3] 설화수 윤조에센스 사용방법 알려줘
    ❌ intent_accuracy: 0.000
    ❌ agent_relevance: 0.000
    ❌ method_relevance: 0.000
    ✅ groundedness: 1.000
    ✅ coherence: 1.000
    ✅ relevance: 1.000
       expected_intent: product_search
       predicted_intent: beauty

[4] 라네즈 워터 슬리핑 마스크 가격이 얼마야?
    ✅ intent_accuracy: 1.000
    ✅ agent_relevance: 1.000
    ✅ method_relevance: 1.000
    ❌ groundedness: 0.200
    ❌ co

## Part 7: Evaluation Dashboard (HTML)

Generate a self-contained HTML dashboard that visualizes evaluation results. This serves as a local alternative to the Foundry portal and can be shared as a standalone report.

The dashboard includes:

- **Summary cards** with average scores per evaluator
- **Per-row results table** with pass/fail indicators
- **Metadata** (eval ID, run ID, model, timestamp)
- **Direct link** to the Foundry portal report

In [ ]:
# Generate HTML evaluation dashboard
DASHBOARD_PATH = "./log/eval_dashboard.html"


def _score_color(score: float) -> str:
    """Return CSS color based on score value."""
    if score >= 0.8:
        return "#22c55e"
    if score >= 0.5:
        return "#eab308"
    return "#ef4444"


def _generate_dashboard_html(
    summary: dict,
    rows: list,
    eval_metrics: dict,
) -> str:
    """Generate self-contained HTML dashboard."""
    report_url = summary.get("report_url", "")
    eval_id = summary.get("eval_id", "N/A")
    run_id = summary.get("run_id", "N/A")
    ts = summary.get("timestamp", "")

    # Build metric cards
    cards_html = ""
    for name, info in eval_metrics.items():
        avg = info["avg"]
        count = info["count"]
        color = _score_color(avg)
        cards_html += f"""
        <div class="card">
          <div class="card-title">{name}</div>
          <div class="card-score" style="color:{color}">
            {avg:.3f}
          </div>
          <div class="card-sub">n={count}</div>
        </div>"""

    # Build results table rows
    table_rows = ""
    for i, row in enumerate(rows, 1):
        query = row.get("query", "")[:80]
        cells = f"<td>{i}</td><td>{query}</td>"
        for name in eval_metrics:
            val = row.get(name)
            if val is not None:
                color = _score_color(val)
                icon = "✅" if val >= 0.5 else "❌"
                cells += (
                    f'<td style="color:{color}">'
                    f"{icon} {val:.2f}</td>"
                )
            else:
                cells += "<td>—</td>"
        table_rows += f"<tr>{cells}</tr>\n"

    # Table header
    metric_headers = "".join(
        f"<th>{n}</th>" for n in eval_metrics
    )

    portal_link = ""
    if report_url:
        portal_link = (
            f'<a href="{report_url}" target="_blank"'
            f' class="portal-link">'
            f"Open in Foundry Portal →</a>"
        )

    return f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>SK Pipeline Evaluation Dashboard</title>
<style>
  * {{ margin: 0; padding: 0; box-sizing: border-box; }}
  body {{
    font-family: -apple-system, BlinkMacSystemFont,
      'Segoe UI', Roboto, sans-serif;
    background: #0f172a; color: #e2e8f0;
    padding: 2rem;
  }}
  .header {{
    text-align: center; margin-bottom: 2rem;
    border-bottom: 1px solid #334155;
    padding-bottom: 1rem;
  }}
  .header h1 {{ color: #60a5fa; font-size: 1.8rem; }}
  .header .meta {{
    color: #94a3b8; font-size: 0.85rem;
    margin-top: 0.5rem;
  }}
  .portal-link {{
    display: inline-block; margin-top: 1rem;
    padding: 0.5rem 1.5rem; background: #2563eb;
    color: white; text-decoration: none;
    border-radius: 6px; font-weight: 600;
  }}
  .portal-link:hover {{ background: #1d4ed8; }}
  .cards {{
    display: flex; gap: 1rem; flex-wrap: wrap;
    justify-content: center; margin: 2rem 0;
  }}
  .card {{
    background: #1e293b; border-radius: 12px;
    padding: 1.5rem 2rem; min-width: 160px;
    text-align: center; border: 1px solid #334155;
  }}
  .card-title {{
    color: #94a3b8; font-size: 0.8rem;
    text-transform: uppercase; letter-spacing: 0.05em;
  }}
  .card-score {{
    font-size: 2rem; font-weight: 700;
    margin: 0.5rem 0;
  }}
  .card-sub {{ color: #64748b; font-size: 0.75rem; }}
  table {{
    width: 100%; border-collapse: collapse;
    margin-top: 1rem; font-size: 0.85rem;
  }}
  th {{
    background: #1e293b; color: #94a3b8;
    padding: 0.75rem; text-align: left;
    border-bottom: 2px solid #334155;
    text-transform: uppercase; font-size: 0.75rem;
  }}
  td {{
    padding: 0.6rem 0.75rem;
    border-bottom: 1px solid #1e293b;
  }}
  tr:hover {{ background: #1e293b; }}
  .section-title {{
    color: #60a5fa; font-size: 1.2rem;
    margin: 2rem 0 1rem; font-weight: 600;
  }}
</style>
</head>
<body>
<div class="header">
  <h1>SK Pipeline Evaluation Dashboard</h1>
  <div class="meta">
    Eval ID: {eval_id} &nbsp;|&nbsp;
    Run ID: {run_id} &nbsp;|&nbsp;
    Model: {MODEL_DEPLOYMENT} &nbsp;|&nbsp;
    Timestamp: {ts}
  </div>
  {portal_link}
</div>
<div class="cards">{cards_html}</div>
<div class="section-title">Per-Row Results</div>
<table>
  <thead>
    <tr><th>#</th><th>Query</th>{metric_headers}</tr>
  </thead>
  <tbody>{table_rows}</tbody>
</table>
</body>
</html>"""


html_content = _generate_dashboard_html(
    summary=eval_summary_payload,
    rows=eval_rows,
    eval_metrics=eval_summary_payload["metrics"],
)

with open(DASHBOARD_PATH, "w", encoding="utf-8") as f:
    f.write(html_content)

print(f"✅ Dashboard saved to {DASHBOARD_PATH}")
print(f"   File size: {len(html_content):,} bytes")

✅ Dashboard saved to ./log/eval_dashboard.html
   File size: 5,224 bytes


In [ ]:
# Display HTML dashboard inline in notebook
from IPython.display import HTML, display

#display(HTML(html_content))

# Print Foundry portal links
print("\n" + "=" * 60)
print("🔗 Azure AI Foundry Links")
print("=" * 60)
if eval_run and hasattr(eval_run, "report_url") and eval_run.report_url:
    print(f"\n   Report URL:\n   {eval_run.report_url}")
else:
    print("\n   ⚠️ No Foundry report URL (local mode).")
    print(
        "   → Fix: Assign 'Cognitive Services "
        "Contributor' + 'Azure AI Developer' "
        "on the Foundry resource."
    )
    print(
        "   → Then re-run Part 5 to create the "
        "Foundry eval run."
    )
print(f"\n   Eval ID:  {eval_object.id}")
print(f"   Run ID:   {eval_run.id if eval_run else 'N/A'}")

print(f"   Mode:     {'Foundry' if foundry_mode else 'Local'}")
print(f"\n   📄 Dashboard: {DASHBOARD_PATH}")


🔗 Azure AI Foundry Links

   ⚠️ No Foundry report URL (local mode).
   → Fix: Assign 'Cognitive Services Contributor' + 'Azure AI Developer' on the Foundry resource.
   → Then re-run Part 5 to create the Foundry eval run.

   Eval ID:  eval_d4384870570742bd81d5dab1aca4cc61
   Run ID:   N/A
   Mode:     Local

   📄 Dashboard: ./log/eval_dashboard.html


## Wrap-up

### Evaluation Pipeline (Azure AI Foundry Evals API)

```text
golden_user_query_list.jsonl
         │
         ▼
POST /chat (SK Backend) → llm_result_list.jsonl
         │
         ▼
project_client.beta.evaluators.create_version()
  → Register code-based evaluators in Foundry Catalog
         │
         ▼
openai_client.evals.create() + evals.runs.create()
  → Run 6 evaluators (3 custom + 3 builtin) server-side
         │
         ├──► Steps 1-3: Exact-match (intent, agent, method)
         └──► Step 4: LLM-based (groundedness, coherence, relevance)
         │
         ▼
report_url → Foundry Portal + HTML Dashboard
```

### Key APIs Used

| SDK | Class/Method | Purpose |
|-----|-------------|---------|
| `azure-ai-projects>=2.0.1` | `AIProjectClient` | Foundry project client |
| `azure-ai-projects` | `project_client.beta.evaluators.create_version()` | Register custom evaluators |
| `openai` (via Foundry) | `openai_client.evals.create()` | Create evaluation group |
| `openai` (via Foundry) | `openai_client.evals.runs.create()` | Run evaluation |
| `openai` (via Foundry) | `openai_client.evals.runs.output_items.list()` | Retrieve per-row results |

### Cleanup

To delete the registered evaluators and evaluation from the Foundry Catalog:

```python
# Delete custom evaluators
for name, reg in registered_evaluators.items():
    project_client.beta.evaluators.delete_version(
        name=reg.name, version=reg.version,
    )
# Delete evaluation
openai_client.evals.delete(eval_id=eval_object.id)
```

## Additional Resources

- [Azure AI Projects SDK 2.0 — Evaluation Samples](https://github.com/Azure/azure-sdk-for-python/tree/main/sdk/ai/azure-ai-projects/samples/evaluations)
- [Code-Based Custom Evaluators](https://github.com/Azure/azure-sdk-for-python/blob/main/sdk/ai/azure-ai-projects/samples/evaluations/sample_eval_catalog_code_based_evaluators.py)
- [Prompt-Based Custom Evaluators](https://github.com/Azure/azure-sdk-for-python/blob/main/sdk/ai/azure-ai-projects/samples/evaluations/sample_eval_catalog_prompt_based_evaluators.py)
- [Model Evaluation Sample](https://github.com/Azure/azure-sdk-for-python/blob/main/sdk/ai/azure-ai-projects/samples/evaluations/sample_model_evaluation.py)
- [Builtin Evaluators with Traces](https://github.com/Azure/azure-sdk-for-python/blob/main/sdk/ai/azure-ai-projects/samples/evaluations/sample_evaluations_builtin_with_traces.py)
- [Azure AI Foundry Portal](https://ai.azure.com/)
- [Semantic Kernel OpenTelemetry](https://learn.microsoft.com/en-us/semantic-kernel/concepts/enterprise-readiness/observability)